# PCA Series | Chapter 3: Probabilistic PCA (PPCA)

> **Chapter Goal:** Reframe PCA as a latent variable generative model. Standard PCA is a geometric algorithm with no probabilistic model — it can't handle missing data, can't quantify uncertainty, and can't tell you the likelihood of a new point. Probabilistic PCA (PPCA) fixes all of this by modeling data as latent codes mapped to observations through Gaussian noise. We derive the model, its MLE solution, and prove it contains standard PCA as a special case.

---

## 1. Motivation: Why Standard PCA is Limited

Standard PCA is purely geometric — it minimizes reconstruction error. It has no notion of:

| Limitation | Consequence |
| :--- | :--- |
| **No noise model** | Can't distinguish signal from noise formally; $k$ is chosen heuristically |
| **No likelihood** | Can't compare PCA models using BIC/AIC; can't test if two datasets share the same PCA structure |
| **Can't handle missing data** | One missing value makes the entire data point unusable |
| **No uncertainty** | Gives point estimates for $k$, for scores, for PCs — no confidence intervals |
| **Not generative** | Can't sample new data points from the model |

PPCA (Tipping & Bishop, 1999) solves all of these by embedding PCA in a probabilistic framework.

---

## 2. The PPCA Generative Model

### **Model Specification**
We assume data $\vec{x} \in \mathbb{R}^d$ is generated from a $k$-dimensional latent code $\vec{z} \in \mathbb{R}^k$ ($k \ll d$) through a linear map plus Gaussian noise:

$$\boxed{\vec{x} = W\vec{z} + \vec{\mu} + \vec{\epsilon}}$$

with priors:
$$\vec{z} \sim \mathcal{N}(\vec{0}, I_k) \quad \text{(latent code: isotropic Gaussian)}$$
$$\vec{\epsilon} \sim \mathcal{N}(\vec{0}, \sigma^2 I_d) \quad \text{(isotropic Gaussian noise)}$$

Parameters:
- $W \in \mathbb{R}^{d \times k}$: **factor loading matrix** (maps latent to observed)
- $\vec{\mu} \in \mathbb{R}^d$: **mean vector**
- $\sigma^2 \in \mathbb{R}_+$: **noise variance** (isotropic — same in all directions)

### **The Marginal Distribution of $\vec{x}$**
Marginalizing over $\vec{z}$ (integrating out the latent variables):
$$\vec{x} \sim \mathcal{N}(\vec{\mu},\ C) \quad \text{where} \quad C = WW^T + \sigma^2 I_d$$

The observed covariance $C$ decomposes into:
- $WW^T$: a rank-$k$ matrix (the structured signal part)
- $\sigma^2 I_d$: isotropic noise (added uniformly to every direction)

### **The Posterior: What Is $\vec{z}$ Given $\vec{x}$?**
Using Bayes' theorem (both are Gaussian → posterior is Gaussian):
$$p(\vec{z}|\vec{x}) = \mathcal{N}(\vec{z};\ M^{-1}W^T(\vec{x}-\vec{\mu}),\ \sigma^2 M^{-1})$$

where $M = W^T W + \sigma^2 I_k \in \mathbb{R}^{k \times k}$.

The **posterior mean** $M^{-1}W^T(\vec{x}-\vec{\mu})$ is the **latent code estimate** for observation $\vec{x}$. This is the reconstruction in the latent space.

---

## 3. Maximum Likelihood Estimation of Parameters

### **Log-Likelihood**
For $N$ i.i.d. observations $\{\vec{x}_1, ..., \vec{x}_N\}$ with $\vec{x}_i \sim \mathcal{N}(\vec{\mu}, C)$, $C = WW^T + \sigma^2 I$:
$$\log p(X | W, \vec{\mu}, \sigma^2) = -\frac{N}{2}\left[d\log(2\pi) + \log|C| + \text{tr}(C^{-1}\hat{\Sigma})\right]$$

where $\hat{\Sigma} = \frac{1}{N}\sum_i (\vec{x}_i - \vec{\mu})(\vec{x}_i - \vec{\mu})^T$ is the sample covariance.

### **Closed-Form MLE Solution** (Tipping & Bishop, 1999)

Let $\hat{\Sigma} = U_d \Lambda_d U_d^T$ be the eigendecomposition of the sample covariance (sorted descending). The MLE solution for $W$ is:

$$\boxed{W_{\text{MLE}} = U_k (\Lambda_k - \sigma^2 I_k)^{1/2} R}$$

where:
- $U_k = [\vec{u}_1 | \cdots | \vec{u}_k]$ — top-$k$ eigenvectors of $\hat{\Sigma}$ (same as PCA directions!)
- $\Lambda_k = \text{diag}(\lambda_1, ..., \lambda_k)$ — top-$k$ eigenvalues
- $R$ — any $k \times k$ rotation matrix (arbitrary — does not affect the model)

The MLE for $\sigma^2$:
$$\boxed{\sigma^2_{\text{MLE}} = \frac{1}{d-k} \sum_{j=k+1}^d \lambda_j = \frac{\text{total discarded variance}}{d-k}}$$

This is the **average variance in the discarded PCA directions** — exactly the noise variance estimate used in the Marchenko-Pastur sense.

---

## 4. The $\sigma^2 \to 0$ Limit: PPCA → Standard PCA

As $\sigma^2 \to 0$ (no noise):

- $W_{\text{MLE}} \to U_k \Lambda_k^{1/2} R$ — columns of $W$ span the same subspace as the top-$k$ PCs.
- Posterior covariance $\sigma^2 M^{-1} \to 0$ — the latent code becomes a point estimate (no uncertainty).
- Posterior mean $\to (W^T W)^{-1} W^T (\vec{x} - \vec{\mu})$ = orthogonal projection of $\vec{x} - \vec{\mu}$ onto the column space of $W$ = **standard PCA projection**.

Standard PCA is the **noiseless limit** of PPCA. The PPCA generalization adds the noise model $\sigma^2$ which regularizes the projection and enables all the probabilistic features.

---

## 5. The EM Algorithm for PPCA

When $d$ is large, computing the MLE requires eigendecomposition of $\hat{\Sigma} \in \mathbb{R}^{d \times d}$. The **EM algorithm** avoids this by iteratively updating $W$ and $\sigma^2$ using the small $k \times k$ matrices.

### **E-Step: Compute Expected Latent Statistics**
For each data point $\vec{x}_i$, compute the expected latent code and second moment:
$$\langle \vec{z}_i \rangle = M^{-1} W^T (\vec{x}_i - \vec{\mu}) \in \mathbb{R}^k$$
$$\langle \vec{z}_i \vec{z}_i^T \rangle = \sigma^2 M^{-1} + \langle \vec{z}_i \rangle \langle \vec{z}_i \rangle^T \in \mathbb{R}^{k \times k}$$

where $M = W^T W + \sigma^2 I_k$ (only $k \times k$ — cheap to invert!).

### **M-Step: Update Parameters**
$$W_{\text{new}} = \left[\sum_i (\vec{x}_i - \vec{\mu}) \langle \vec{z}_i \rangle^T\right] \left[\sum_i \langle \vec{z}_i \vec{z}_i^T \rangle\right]^{-1}$$

$$\sigma^2_{\text{new}} = \frac{1}{Nd} \sum_i \left[\|\vec{x}_i - \vec{\mu}\|^2 - 2\langle\vec{z}_i\rangle^T W^T (\vec{x}_i - \vec{\mu}) + \text{tr}(\langle\vec{z}_i\vec{z}_i^T\rangle W^T W)\right]$$

$$\vec{\mu}_{\text{new}} = \frac{1}{N}\sum_i \vec{x}_i \quad \text{(unchanged from standard estimate)}$$

### **Advantages of EM over Direct MLE**
- **Works for large $d$:** Only $k \times k$ matrix inversions (not $d \times d$).
- **Handles missing data:** In E-step, only observed dimensions of $\vec{x}_i$ contribute.
- **Guaranteed monotone likelihood:** Each iteration increases the log-likelihood.
- **Can be run online (streaming):** Update E and M steps one data point at a time.

---

## 6. Handling Missing Data

One of PPCA's strongest advantages over standard PCA: **native handling of missing values**.

### **How It Works**
For data point $\vec{x}_i$ with some dimensions missing, let $\mathcal{O}_i$ = set of observed dimensions and $\mathcal{M}_i$ = missing dimensions.

In the E-step, instead of computing $p(\vec{z}_i | \vec{x}_i)$, compute $p(\vec{z}_i | \vec{x}_{i, \mathcal{O}_i})$ — the posterior given only the **observed** dimensions.

This is still Gaussian (since everything is linear-Gaussian), with:
$$M_i = W_{\mathcal{O}_i}^T W_{\mathcal{O}_i} + \sigma^2 I_k \quad \text{(only observed rows of } W \text{)}$$

The M-step uses only the observed dimensions for each data point — seamlessly averaging over the missing ones.

### **Imputation**
After fitting PPCA, imputed values for missing dimensions:
$$\hat{x}_{i,j} = (W \langle \vec{z}_i \rangle + \vec{\mu})_j \quad \text{for } j \in \mathcal{M}_i$$

Standard PCA cannot do this at all — one missing value forces you to either drop the entire row or use mean imputation before running PCA.

---

## 7. PPCA vs Factor Analysis

Factor Analysis (FA) has the same generative structure as PPCA but relaxes the isotropic noise assumption:

| Feature | PPCA | Factor Analysis |
| :--- | :--- | :--- |
| Noise model | $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$ — equal variance in all dims | $\epsilon \sim \mathcal{N}(0, \Psi)$ — diagonal $\Psi$, different variance per dim |
| Parameters | $W, \vec{\mu}, \sigma^2$ (one scalar for noise) | $W, \vec{\mu}, \Psi$ ($d$ noise variances to fit) |
| PCA connection | $\sigma^2 \to 0$ gives standard PCA | No such direct connection |
| Invariance | Rotationally invariant — $WR$ gives same model | Rotationally invariant only in latent space |
| Uniqueness | $W$ determined up to rotation $R$ | $W$ determined up to rotation $R$ (same ambiguity) |
| Flexibility | Less flexible (isotropic noise) | More flexible (anisotropic noise) |
| Interpretability | Noise $\sigma^2$ has single intuitive value | Each feature can have its own noise level |

**When to use FA over PPCA:** When different features have very different noise levels (e.g., a mix of very reliable and very noisy sensors).

---

## 8. Interview Q&A

**Q1: What is Probabilistic PCA?**
> PPCA is a latent variable model where observed $d$-dimensional data is assumed generated from a $k$-dimensional Gaussian latent code $\vec{z} \sim \mathcal{N}(0, I)$ via $\vec{x} = W\vec{z} + \vec{\mu} + \epsilon$, with isotropic noise $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$. Marginalizing over $\vec{z}$ gives $\vec{x} \sim \mathcal{N}(\vec{\mu}, WW^T + \sigma^2 I)$.

**Q2: How does PPCA reduce to standard PCA?**
> In the $\sigma^2 \to 0$ limit, the noise vanishes and the posterior becomes a point at the standard PCA projection. The MLE for $W$ converges to $W = U_k \Lambda_k^{1/2} R$ — the columns span the same subspace as the top-$k$ PCA eigenvectors.

**Q3: What is the MLE for $\sigma^2$ in PPCA?**
> $\sigma^2_{\text{MLE}} = \frac{1}{d-k}\sum_{j=k+1}^d \lambda_j$ — the average variance in the $d-k$ discarded PCA directions. This is a meaningful and data-driven noise estimate.

**Q4: How does PPCA handle missing data?**
> In PPCA's EM algorithm, the E-step computes the latent posterior using only the observed dimensions of each data point. The M-step updates $W$ and $\sigma^2$ using these per-point posteriors. Missing dimensions are implicitly marginalized over rather than imputed. After fitting, they can be explicitly imputed as $\hat{x}_{i,j} = (W\langle\vec{z}_i\rangle + \vec{\mu})_j$.

**Q5: What is the difference between PPCA and Factor Analysis?**
> Both use the same generative form $\vec{x} = W\vec{z} + \vec{\mu} + \epsilon$. The difference is the noise model: PPCA uses isotropic noise $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$ (one scalar), while FA uses anisotropic noise $\epsilon \sim \mathcal{N}(0, \Psi)$ with a diagonal $\Psi$ (one noise variance per feature). FA is more flexible but has more parameters.

**Q6: Why use EM for PPCA instead of direct MLE?**
> Direct MLE requires eigendecomposing the $d \times d$ sample covariance matrix, which is infeasible for large $d$. EM only requires inverting the $k \times k$ matrix $M = W^TW + \sigma^2 I$ — much cheaper. EM also naturally extends to missing data.

**Q7: Can you choose $k$ automatically in PPCA?**
> Yes. Using Bayesian PPCA with Automatic Relevance Determination (ARD): place a Gamma prior on each column of $W$ that encourages sparsity. During inference, irrelevant latent dimensions get pruned (their prior variance goes to zero). The effective $k$ is determined by the data.

---

In [ ]:
# ─── CELL 1: PPCA via EM Algorithm — From Scratch ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

class PPCA:
    """
    Probabilistic PCA via the EM algorithm.
    Model: x = W z + mu + eps,  z ~ N(0, I_k),  eps ~ N(0, sigma^2 I_d)
    """
    def __init__(self, n_components, n_iter=200, tol=1e-6, random_state=None):
        self.k = n_components
        self.n_iter = n_iter
        self.tol = tol
        self.rng = np.random.default_rng(random_state)

    def fit(self, X):
        N, d = X.shape
        k = self.k
        
        # Initialize
        self.mu_ = X.mean(axis=0)
        Xc = X - self.mu_
        self.W_ = self.rng.standard_normal((d, k)) * 0.01
        self.sigma2_ = np.var(Xc)
        log_likelihoods = []

        for iteration in range(self.n_iter):
            W, s2 = self.W_, self.sigma2_

            # ── E-step ──────────────────────────────────────────────────────────
            # M = W^T W + sigma^2 I_k  (k x k)
            M = W.T @ W + s2 * np.eye(k)
            M_inv = np.linalg.inv(M)
            # Expected latent codes: (k x N)
            Ez = M_inv @ W.T @ Xc.T          # k x N
            # Expected second moments: (k x k)
            Ezz = N * s2 * M_inv + Ez @ Ez.T  # k x k

            # ── M-step ──────────────────────────────────────────────────────────
            # Update W
            W_new = (Xc.T @ Ez.T) @ np.linalg.inv(Ezz)  # d x k
            # Update sigma^2
            s2_new = (np.sum(Xc**2)
                      - 2 * np.trace(Ez @ Xc @ W_new)
                      + np.trace(Ezz @ W_new.T @ W_new)) / (N * d)

            # ── Log-likelihood ──────────────────────────────────────────────────
            C = W_new @ W_new.T + s2_new * np.eye(d)
            sign, logdet = np.linalg.slogdet(C)
            try:
                C_inv = np.linalg.solve(C, np.eye(d))
                ll = -0.5 * N * (d * np.log(2*np.pi) + logdet
                                 + np.trace(C_inv @ (Xc.T @ Xc) / N))
            except:
                ll = float('-inf')
            log_likelihoods.append(ll)

            # Check convergence
            self.W_, self.sigma2_ = W_new, max(s2_new, 1e-10)
            if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < self.tol:
                print(f"Converged at iteration {iteration+1}")
                break

        self.log_likelihoods_ = log_likelihoods
        self.M_ = self.W_.T @ self.W_ + self.sigma2_ * np.eye(k)
        return self

    def transform(self, X):
        """Compute posterior mean of latent codes."""
        Xc = X - self.mu_
        M_inv = np.linalg.inv(self.M_)
        return (M_inv @ self.W_.T @ Xc.T).T  # N x k

    def reconstruct(self, X):
        Z = self.transform(X)  # N x k
        return Z @ self.W_.T + self.mu_  # N x d

# Test on synthetic data
np.random.seed(42)
N, d, k_true, sigma2_true = 300, 20, 3, 0.5

W_gen = np.random.randn(d, k_true)
z = np.random.randn(N, k_true)
X = z @ W_gen.T + np.sqrt(sigma2_true) * np.random.randn(N, d)
X += np.array([1.0]*d)  # non-zero mean

ppca = PPCA(n_components=k_true, n_iter=500, random_state=0).fit(X)

print(f"True σ²: {sigma2_true:.3f}")
print(f"PPCA σ²: {ppca.sigma2_:.3f}")

# MLE sigma^2 from PCA eigenvalues (should match EM result)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Xc = X - X.mean(axis=0)
pca_full = PCA().fit(Xc)
sigma2_mle = pca_full.explained_variance_[k_true:].mean()
print(f"PCA-based σ² MLE: {sigma2_mle:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(ppca.log_likelihoods_, color='steelblue', lw=2)
axes[0].set_xlabel('EM Iteration'); axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title('PPCA EM: Log-Likelihood Convergence'); axes[0].grid(alpha=0.3)

# Compare PC subspaces
pca_k = PCA(n_components=k_true).fit(Xc)
W_pca = pca_k.components_.T  # d x k
W_ppca = ppca.W_

# Subspace alignment: how much of PCA subspace is captured by PPCA subspace?
Q_pca, _ = np.linalg.qr(W_pca)
Q_ppca, _ = np.linalg.qr(W_ppca)
# Squared singular values of Q_pca^T Q_ppca = alignment
alignment = np.linalg.svd(Q_pca.T @ Q_ppca, compute_uv=False)**2
axes[1].bar(range(1, k_true+1), alignment, color='darkorange', alpha=0.8)
axes[1].set_xlabel('Singular value index'); axes[1].set_ylabel('Alignment (1=perfect)')
axes[1].set_ylim(0, 1.1); axes[1].set_title('PPCA vs PCA Subspace Alignment\n(1.0 = identical subspaces)')
axes[1].grid(alpha=0.3, axis='y')
axes[1].axhline(1.0, color='green', ls='--', lw=1.5)

plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: PPCA for Missing Data Imputation — vs Mean Imputation ─────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

np.random.seed(42)
N, d, k_true = 200, 10, 2

# Generate low-rank data (true k=2)
W_true = np.random.randn(d, k_true)
Z_true = np.random.randn(N, k_true)
X_clean = Z_true @ W_true.T + 0.3 * np.random.randn(N, d)

# Introduce missing values (MCAR — missing completely at random)
missing_rate = 0.20  # 20% missing
X_missing = X_clean.copy()
missing_mask = np.random.rand(N, d) < missing_rate
X_missing[missing_mask] = np.nan

# Strategy 1: Mean imputation + PCA
mean_imputer = SimpleImputer(strategy='mean')
X_mean_imp = mean_imputer.fit_transform(X_missing)
pca_mean = PCA(n_components=k_true).fit_transform(X_mean_imp)

# Strategy 2: Iterative PPCA-style imputation
def ppca_impute(X_obs, k, n_iter=50):
    """Simple iterative PPCA imputation: alternate between PCA and fill-in."""
    X = X_obs.copy()
    # Initialize missing with column means
    col_means = np.nanmean(X, axis=0)
    missing = np.isnan(X)
    for j in range(X.shape[1]):
        X[missing[:, j], j] = col_means[j]
    
    for _ in range(n_iter):
        # Fit PCA on current data
        Xc = X - X.mean(axis=0)
        pca = PCA(n_components=k).fit(Xc)
        # Reconstruct and only update missing positions
        X_recon = pca.inverse_transform(pca.transform(Xc)) + X.mean(axis=0)
        X[missing] = X_recon[missing]
    return X, pca

X_ppca_imp, pca_ppca = ppca_impute(X_missing, k=k_true)
pca_ppca_scores = pca_ppca.transform(X_ppca_imp - X_ppca_imp.mean(axis=0))

# Evaluate: MSE of imputed vs true values
mse_mean = np.mean((X_mean_imp[missing_mask] - X_clean[missing_mask])**2)
mse_ppca = np.mean((X_ppca_imp[missing_mask] - X_clean[missing_mask])**2)

print(f"Missing rate: {missing_rate*100:.0f}%")
print(f"Mean imputation MSE: {mse_mean:.4f}")
print(f"PPCA-iter imputation MSE: {mse_ppca:.4f}")
print(f"PPCA improvement: {(mse_mean - mse_ppca)/mse_mean * 100:.1f}% reduction in error")

# Plot: PC space for both imputation strategies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(pca_mean[:, 0], pca_mean[:, 1], s=15, alpha=0.5, color='steelblue', label='Mean imputed')
ax1.set_title(f'Mean Imputation → PCA\nMSE={mse_mean:.3f}', fontsize=11)
ax1.grid(alpha=0.3); ax1.legend()

ax2.scatter(pca_ppca_scores[:, 0], pca_ppca_scores[:, 1], s=15, alpha=0.5, color='darkorange', label='PPCA imputed')
ax2.set_title(f'PPCA Iterative Imputation → PCA\nMSE={mse_ppca:.3f}', fontsize=11)
ax2.grid(alpha=0.3); ax2.legend()

plt.suptitle('Missing Data Imputation: Mean vs PPCA Iterative', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()